In [1]:
import os
import sys
import json

os.chdir("/home/kaariaa3/mscthesis/")
sys.path.append("./src/")  # Add module directory to path

# Set environment variables before importing transformers
os.environ["HUGGINGFACE_HUB_CACHE"] = "/scratch/shareddata/dldata/huggingface-hub-cache/hub" # Load local models
os.environ["TRANSFORMERS_OFFLINE"] = "1" 
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_HOME"] = f"{os.environ["WRKDIR"]}/.cache/huggingface" # Cache in work directory

import transformers
import numpy as np
from datasets import load_dataset

from utils.helpers import get_system_prompt, get_task_type, get_default_response, make_prompt, parse_output

from utils.constants import PIPE_MAX_NEW_TOKENS, MODEL_TEMPERATURE, BATCH_SIZE, PIPE_RETURN_FULL_TEXT, CONCEPT_TO_CHAPTER_MAPPING

In [2]:
LABELS = ["yes", "no"] # Labels per column
EVAL_COLS = ['The exercise description matched the selected theme (Yes/No)',
       'The exercise description matched the selected topic (Yes/No)',
            "Included concepts that were too advanced (Yes/No)"]
PRED_COLS = ['augmentedProblemDescription', 'augmentedExampleSolution']

DEFAULT_DATA = r"./data/final_dataset.csv"
DEFAULT_MODEL = "Qwen/Qwen2.5-14B-Instruct"

In [3]:
task = "detect"
n_rows = 10

dataset = load_dataset("csv", data_files=DEFAULT_DATA, split="train", sep=";")
dataset = dataset.shuffle(seed=42)
if n_rows is not None and n_rows > 0:
    dataset = dataset.select(range(n_rows))

demonstrations_rng = np.random.default_rng(seed=10)
n_demonstrations = 1

# Make user and system prompts
dataset = dataset.map(lambda row: {"user_prompt": make_prompt(row, task), "system_prompt": get_system_prompt(task, dataset.select(demonstrations_rng.integers(low=0, high=dataset.num_rows, size=n_demonstrations)))})

In [4]:
for row in dataset:
    print(row["system_prompt"])

Use the demonstrations below as examples on how to answer the questionDemonstration:

Theme: outdoor activities
Topic: snowboarding
Concept: user input

--- PROBLEM DESCRIPTION ---
Write a program that asks the user for their favorite Pablo Picasso painting. After this, the program prints a message to the user ‘Wow! I love the painting!’, where painting is the painting entered by the user. For example, with the input `Guernica`, the program output is as follows:

```
What is your favorite Pablo Picasso painting?
Guernica
Wow! I love the Guernica!
```

Similarly, if the user enters the painting `Les Demoiselles d'Avignon`, the program output is as follows:

```
What is your favorite Pablo Picasso painting?
Les Demoiselles d\u0027Avignon
Wow! I love the Les Demoiselles d\u0027Avignon!
```

--- EXAMPLE SOLUTION ---
{"code": "import 'dart:io';main() {  print('What is your favorite Pablo Picasso painting?');  String painting = '';  while (painting.isEmpty) {    painting = stdin.readLineSync

In [4]:
params = {
    "model": DEFAULT_MODEL,
    "device_map": 0,  # Force GPU
    "max_new_tokens": PIPE_MAX_NEW_TOKENS,
    "temperature": MODEL_TEMPERATURE,
}

pipeline = transformers.pipeline("text-generation", **params)
pipeline.tokenizer.pad_token = pipeline.tokenizer.eos_token
pipeline.model.config.pad_token_id = pipeline.model.config.eos_token_id

2026-03-18 16:19:48.314571: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

Device set to use cuda:0


In [6]:
default_response = get_default_response(task)
results = {key: [] for key in default_response.keys()}

batch_size = BATCH_SIZE
prompts = dataset["user_prompt"]
system_prompts = dataset["system_prompt"]
for i in range(0, len(prompts), batch_size):
    
    # Build batched conversation inputs
    batch_prompts = zip(system_prompts[i : i + batch_size], prompts[i : i + batch_size])
    batch_inputs = [
        [
            {"role": "system", "content": sp},
            {"role": "user", "content": up},
        ]
        for sp, up in batch_prompts
    ]

    # Single batched forward pass
    outputs = pipeline(
        batch_inputs,
        return_full_text=PIPE_RETURN_FULL_TEXT,
    )

    # Process each output in the batch
    for output in outputs:
        text = output[0]["generated_text"]
        parsed = parse_output(text)

        for key, value in default_response.items():
            results[key].append(json.dumps(parsed.get(key, value)))
print("done")

done


In [7]:
import pandas as pd

data = dataset.to_pandas()

def calculate_accuracy(df, predictions):

    # Ground truth columns
    gt_theme = df['The exercise description matched the selected theme (Yes/No)']
    gt_topic = df['The exercise description matched the selected topic (Yes/No)']
    gt_concept = df['Included concepts that were too advanced (Yes/No)']

    # Normalize values
    def normalize(series):
        return series.astype(str).str.strip('"').str.lower()

    gt_theme = normalize(gt_theme)
    gt_topic = normalize(gt_topic)
    gt_concept = normalize(gt_concept)

    pred_theme = normalize(pd.Series(predictions['themeCorrect']))
    pred_topic = normalize(pd.Series(predictions['topicCorrect']))
    pred_concept = normalize(pd.Series(predictions['usesAdditionalConcepts']))
    
    # Accuracy calculation
    theme_acc = (gt_theme == pred_theme)
    topic_acc = (gt_topic == pred_topic)
    concept_acc = (gt_concept == pred_concept)

    return {
        "theme_accuracy": theme_acc.mean(),
        "topic_accuracy": topic_acc.mean(),
        "concept_accuracy": concept_acc.mean(),
        "total_accuracy": (theme_acc & topic_acc & concept_acc).mean()
    }

In [13]:
def print_row(row):
    print("##################")
    print("#  Ground-truth  #")
    print("##################")
    print("Theme correct: " + row[EVAL_COLS[0]])
    print("Topic correct: " + row[EVAL_COLS[1]])
    print("Uses advanced concepts: " + row[EVAL_COLS[2]])
    print("Original concept: " + row["concept"])
    print(row["system_prompt"])
    print(row["user_prompt"])

    print("##################")
    print("#  Prediction    #")
    print("##################")
    print("Theme correct: " + results["themeCorrect"][i])
    print("Topic correct: " + results["topicCorrect"][i])
    print("Uses advanced concepts: " + results["usesAdditionalConcepts"][i])
    print("Explanation: " + results["explanation"][i])
    print("---------------------------------------------------")
    print()

print_n_rows = 1
if True:
    for i, row in data.iloc[:print_n_rows, :].iterrows():
        print_row(row)

##################
#  Ground-truth  #
##################
Theme correct: yes
Topic correct: yes
Uses advanced concepts: no
Original concept: logical operators
Use the demonstrations below as examples on how to answer the questionDemonstration:

Theme: outdoor activities
Topic: snowboarding
Concept: user input

--- PROBLEM DESCRIPTION ---
Write a program that asks the user for their favorite Pablo Picasso painting. After this, the program prints a message to the user ‘Wow! I love the painting!’, where painting is the painting entered by the user. For example, with the input `Guernica`, the program output is as follows:

```
What is your favorite Pablo Picasso painting?
Guernica
Wow! I love the Guernica!
```

Similarly, if the user enters the painting `Les Demoiselles d'Avignon`, the program output is as follows:

```
What is your favorite Pablo Picasso painting?
Les Demoiselles d\u0027Avignon
Wow! I love the Les Demoiselles d\u0027Avignon!
```

--- EXAMPLE SOLUTION ---
{"code": "import '

In [9]:
calculate_accuracy(data, results)

{'theme_accuracy': np.float64(0.5),
 'topic_accuracy': np.float64(0.7),
 'concept_accuracy': np.float64(0.7),
 'total_accuracy': np.float64(0.3)}